In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import re
import os
from difflib import get_close_matches
# Importar Base
# Leer datos iniciales
ruta_archivo = '/kaggle/input/datasets/geraldinelaverde/inicial-giros/Recursos_del_Sistema_General_Giros_20251018.csv'

df_Giros_inicial = pd.read_csv(ruta_archivo, low_memory=False)

df_Giros_inicial.head()



,Fecha de pago,Identificacion- NIT,Nombre Razon Social,Descripcion,Fuente,Valor Girado,Concepto Pago,Ordenes de Pago
0,2015-01-29,800136069,MUNICIPIO DE FORTUL,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,27.047.767,GIRO SGP SALUD PUBLICA VIG 2014,9207515
1,2015-01-29,890000464,MUNICIPIO DE ARMENIA,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,130.243.583,GIRO SGP SALUD PUBLICA VIG 2014,9207615
2,2015-01-29,899999372,MUNICIPIO DE SIBATE,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,12.600.367,GIRO SGP SALUD PUBLICA VIG 2014,9207715
3,2015-01-29,890980112,MUNICIPIO DE BELLO,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,214.941.350,GIRO SGP SALUD PUBLICA VIG 2014,9207815
4,2015-01-29,890985285,MUNICIPIO DE VEGACHI,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,9.040.907,GIRO SGP SALUD PUBLICA VIG 2014,9207915


In [2]:
# Ver datso divipola
print(os.listdir('/kaggle/input/datasets/nicolasacostaa/municipios-divipola'))

#Importar divipola
# Leer datos iniciales

ruta_davipola = '/kaggle/input/datasets/nicolasacostaa/municipios-divipola/DIVIPOLA_Municipios.xlsx - Municipios.csv'

df_Davipola = pd.read_csv(ruta_davipola, encoding='utf-8')
 
df_Davipola = df_Davipola.rename(columns={
    'doc_dep':      'CÓDIGO DEPARTAMENTO',
    'departamento': 'NOMBRE DEPARTAMENTO',
    'cod_muni':     'CÓDIGO MUNICIPIO',
    'municipio':    'NOMBRE MUNICIPIO',
})
 
# Códigos con ceros a la izquierda
df_Davipola['CÓDIGO DEPARTAMENTO'] = df_Davipola['CÓDIGO DEPARTAMENTO'].astype(str).str.zfill(2)
df_Davipola['CÓDIGO MUNICIPIO']    = df_Davipola['CÓDIGO MUNICIPIO'].astype(str).str.zfill(5)
 
df_Davipola.head()

['DIVIPOLA_Municipios.xlsx', 'DIVIPOLA_Municipios.xlsx - Municipios.csv']


,CÓDIGO DEPARTAMENTO,NOMBRE DEPARTAMENTO,CÓDIGO MUNICIPIO,NOMBRE MUNICIPIO
0,91,AMAZONAS,91263,EL ENCANTO
1,91,AMAZONAS,91405,LA CHORRERA
2,91,AMAZONAS,91407,LA PEDRERA
3,91,AMAZONAS,91430,LA VICTORIA
4,91,AMAZONAS,91001,LETICIA


In [3]:
# ----------------------------
# LIMPIEZA DE TIPOS 
# ----------------------------

# NIT como texto
df_Giros_inicial['Identificacion- NIT'] = df_Giros_inicial['Identificacion- NIT'].astype(str)

# Fecha correcta
df_Giros_inicial['Fecha de pago'] = pd.to_datetime(
    df_Giros_inicial['Fecha de pago'],
    format='%Y-%m-%d',
    errors='coerce'
)

df_Giros_inicial.dtypes

Fecha de pago          datetime64[ns]
Identificacion- NIT            object
Nombre Razon Social            object
Descripcion                    object
Fuente                         object
Valor Girado                   object
Concepto Pago                  object
Ordenes de Pago                object
dtype: object

**FUNCIONES DE NORMALIZACIÓN**

In [4]:
def normalizar_robusto(texto):
    """Convierte texto a ASCII limpio sin tildes, manejando dos situaciones:
    
    1. Texto bien codificado (UTF-8): tildes reales → se eliminan con NFKD
       Ejemplo: "NARIÑO" → "NARINO"
    
    2. Mojibake: el CSV de Giros es UTF-8 guardado y leído como latin-1,
       dejando secuencias como 'Ã\x91' en lugar de 'Ñ'.
       El truco encode('latin1').decode('utf-8') lo repara antes de normalizar.
       Ejemplo: "BRICEÃ\x91O" → "BRICEÑO" → "BRICENO"
    
    Si la reparación falla (bytes mixtos), usa el texto como está y normaliza.
    """
    if pd.isna(texto):
        return ''
    s = str(texto).upper().strip()
    # Intentar reparar mojibake
    try:
        s = s.encode('latin1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass  # texto ya limpio o bytes mixtos irrecuperables → normalizar directo
    # Eliminar diacríticos y convertir a ASCII
    s = unicodedata.normalize('NFKD', s)
    return ' '.join(s.encode('ASCII', 'ignore').decode('utf-8').split())
 

In [5]:
def extraer_dpto_desc(desc):
    """Extrae el nombre de departamento desde la columna Descripcion.
    
    Ejemplos:
      'SGP SALUD PUBLICA DEPARTAMENTO DE SANTANDER'   → 'SANTANDER'
      'SGP SALUD PUBLICA DEPARTAMENTO DEL META'        → 'META'
      'SGP SALUD PUBLICA DEPARTAMENTO DE NARIÃO'       → 'NARINO'  (repara mojibake)
    
    Correcciones de nombres: algunos departamentos aparecen truncados o
    distintos en Giros vs Davipola; se unifican aquí.
    """
    if pd.isna(desc):
        return ''
    m = re.search(r'DEPARTAMENTO\s+DE(?:L)?\s+(.+)$', str(desc).upper())
    if not m:
        return ''
    dpto = normalizar_robusto(m.group(1).strip())
 
    # Correcciones puntuales de discrepancias Giros ↔ Davipola
    correcciones = {
        'GUAJIRA':    'LA GUAJIRA',   # Giros omite el "LA"
        'NARI-O':     'NARINO',       # Giros tiene guion en vez de Ñ
        'SAN ANDRES': 'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA',
    }
    return correcciones.get(dpto, dpto)

In [6]:
# Municipios cuyo nombre termina con el nombre de su departamento.
# Sin esta lista DPTOS_1_PALABRA los recortaba erróneamente.
_NOMBRES_MUNICIPIO_PROTEGIDOS = {
    'PUERTO SANTANDER',         # Norte de Santander
    'PUERTO NARINO',            # Amazonas
    'PUERTO BOYACA',            # Boyacá
    'SANTA FE DE ANTIOQUIA',    # Antioquia
    'CIUDAD BOLIVAR',           # Antioquia / Bolívar (dpto desambigua)
    'SAN JUAN DEL CESAR',       # La Guajira
    'SAN JOSE DEL GUAVIARE',    # Guaviare
    'SAN JACINTO DEL CAUCA',    # Bolívar
    'VILLA DEL ROSARIO',        # Norte de Santander
    'SANTA ROSA DE CABAL',      # Risaralda
    'SANTA ROSA DE OSOS',       # Antioquia
    'PUERTO COLOMBIA',          # Atlántico
    'PUERTO RICO',              # Caquetá / Meta
    'SAN ANDRES DE TUMACO',     # Nariño
    'SAN JUAN DE URABA',        # Antioquia
    'SAN PEDRO DE URABA',       # Antioquia
    'SANTA BARBARA DE PINTO',   # Magdalena
    'CARMEN DEL DARIEN',        # Chocó
    'VILLA RICA',               # Cauca
    'MAGANGUE',                 # Bolívar  (termina en 'E', no en dpto — seguro)
    'SAN JACINTO DEL CAUCA',    # Bolívar
}
 
def limpiar_nombre(texto):
    """
    Limpia Nombre Razon Social para usarlo como llave de match.
 
    Correcciones respecto a la versión anterior:
      - Añadido prefijo  r'MUNICIPIO\\s+'  (sin DE) para capturar
        casos como 'MUNICIPIO MAGUIPAYAN'.
      - Añadidos sufijos r'\\s+ALCALDIA\\s+MUNICIPAL$' y r'\\s+MUNICIPAL$'
        para limpiar residuos como 'PACHO ALCALDIA MUNICIPAL' → 'PACHO'.
      - DPTOS_1_PALABRA solo recorta si el nombre completo NO está en
        _NOMBRES_MUNICIPIO_PROTEGIDOS, evitando que 'PUERTO SANTANDER'
        quede como 'PUERTO'.
    """
    if not texto:
        return ''
    texto = normalizar_robusto(str(texto))
 
    # Prefijos a eliminar (más largos primero)
    prefijos = [
        r'MUNICIPIO\s+DE(?:L)?\s+',
        r'MUNICIPIO\s+',                            # FIX C
        r'DEPARTAMENTO\s+ARCHIPIELAGO\s+DE\s+',
        r'DEPARTAMENTO\s+DE(?:L)?\s+',
        r'GOBIERNO\s+DEPARTAMENTAL\s+DE(?:L)?\s+',
        r'ALCALDIA\s+MUNICIPAL\s+DE(?:L)?\s+',
        r'ALCALDIA\s+MUNICIPAL\s+',
        r'ALCALDIA\s+DE(?:L)?\s+',
        r'ALCALDIA\s+',
    ]
    for p in prefijos:
        texto = re.sub(p, '', texto).strip()
 
    # Sufijos a eliminar
    sufijos = [
        r'\s+DPTO\s+\.?\s*DE(?:L)?\s+\w+\.?$',
        r'\s+DEPARTAMENTO\s+DE(?:L)?\s+\w+\.?$',
        r'\s+ALCALDIA\s+MUNICIPAL\.?$',             # FIX B (con punto)
        r'\s+ALCALDIA\s+MUNICIPAL$',                # FIX B
        r'\s+MUNICIPAL$',                           # FIX B (residuo)
        r'\s+ALCALDIA$',
        r'\s+EN\s+REESTRUCTURACION.*$',
        r'\.$',
        r'\s+-\s+\w+$',
    ]
    for s in sufijos:
        texto = re.sub(s, '', texto).strip()
 
    # Quitar departamento de 1 palabra al final SOLO si no está protegido
    DPTOS_1_PALABRA = {
        'ANTIOQUIA', 'ATLANTICO', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA',
        'CASANARE', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAMARCA',
        'GUAINIA', 'GUAVIARE', 'HUILA', 'MAGDALENA', 'META', 'NARINO',
        'PUTUMAYO', 'QUINDIO', 'RISARALDA', 'AMAZONAS', 'SANTANDER',
        'SUCRE', 'TOLIMA', 'VAUPES', 'VICHADA', 'ARAUCA',
    }
    partes = texto.split()
    if (len(partes) >= 2
            and partes[-1] in DPTOS_1_PALABRA
            and texto not in _NOMBRES_MUNICIPIO_PROTEGIDOS):   # FIX A
        texto = ' '.join(partes[:-1]).strip()
 
    return texto

In [7]:
#Aplicar normalización a ambos DataFrames
# En Giros: llave de match + departamento extraído de Descripcion
df_Giros_inicial['Nombre_match'] = df_Giros_inicial['Nombre Razon Social'].apply(limpiar_nombre)
df_Giros_inicial['_dpto_desc']   = df_Giros_inicial['Descripcion'].apply(extraer_dpto_desc)
 
# En Davipola: llave de match + departamento normalizado
df_Davipola['Nombre_match'] = df_Davipola['NOMBRE MUNICIPIO'].apply(normalizar_robusto)
df_Davipola['_dpto_norm']   = df_Davipola['NOMBRE DEPARTAMENTO'].apply(normalizar_robusto)
 
# Corrección: LA GUAJIRA en Davipola se normaliza a "LA GUAJIRA",
# pero extraer_dpto_desc ya devuelve "LA GUAJIRA" tras la corrección → OK
# Verificar rápido
print("Davipola _dpto_norm únicos:", sorted(df_Davipola['_dpto_norm'].unique()))
 
 


Davipola _dpto_norm únicos: ['AMAZONAS', 'ANTIOQUIA', 'ARAUCA', 'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA', 'ATLANTICO', 'BOGOTA, D.C.', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA', 'CASANARE', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAMARCA', 'GUAINIA', 'GUAVIARE', 'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARINO', 'NORTE DE SANTANDER', 'PUTUMAYO', 'QUINDIO', 'RISARALDA', 'SANTANDER', 'SUCRE', 'TOLIMA', 'VALLE DEL CAUCA', 'VAUPES', 'VICHADA']


In [8]:
# MERGE
columnas_deseadas = [
    'Fecha de pago',
    'CÓDIGO DEPARTAMENTO',
    'NOMBRE DEPARTAMENTO',
    'CÓDIGO MUNICIPIO',
    'NOMBRE MUNICIPIO',
    'Identificacion- NIT',
    'Nombre Razon Social',
    'Descripcion',
    'Fuente',
    'Valor Girado',
    'Concepto Pago',
    'Ordenes de Pago',
]
 
COLS_DAVI = ['Nombre_match', '_dpto_norm',
             'CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
             'CÓDIGO MUNICIPIO',    'NOMBRE MUNICIPIO']
 
# ── PASO 1: Cruce directo por Nombre_match ──────────────────────
# Solo municipios con nombre ÚNICO en Colombia (keep=False descarta ambiguos,
# que son resueltos en el Paso 2 con el departamento como desempate).
davi_uniq = (df_Davipola[COLS_DAVI]
             .drop_duplicates(subset='Nombre_match', keep=False))
 
df_Giros_1 = df_Giros_inicial.merge(davi_uniq, on='Nombre_match', how='left')
 
n1 = df_Giros_1['CÓDIGO MUNICIPIO'].notna().sum()
print(f"Paso 1 — nombre único:               {n1:,} filas cruzadas")
 
 
# ── PASO 2: Cruce por (Nombre_match + departamento de Descripcion) ──
# Resuelve ambiguos como "BOLIVAR" (Santander vs Cauca vs Valle del Cauca).
mask2 = df_Giros_1['CÓDIGO MUNICIPIO'].isna()
 
davi_mun_dpto = (df_Davipola[COLS_DAVI]
                 .drop_duplicates(subset=['Nombre_match', '_dpto_norm']))
 
df_paso2 = (df_Giros_1.loc[mask2, ['Nombre_match', '_dpto_desc']]
            .merge(davi_mun_dpto,
                   left_on=['Nombre_match', '_dpto_desc'],
                   right_on=['Nombre_match', '_dpto_norm'],
                   how='left'))
 
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO',    'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[mask2, col] = df_paso2[col].values
 
n2 = df_Giros_1.loc[mask2, 'CÓDIGO MUNICIPIO'].notna().sum()
print(f"Paso 2 — nombre + dpto Descripcion: {n2:,} nuevas filas cruzadas")
 
 
# ── PASO 3: Entidades departamentales ───────────────────────────
# Filas donde el beneficiario ES el departamento o una entidad departamental.
# No tienen municipio → asignamos solo CÓDIGO/NOMBRE DEPARTAMENTO.
# Ejemplos: "DEPARTAMENTO DEL META", "GOBIERNO DEPARTAMENTAL DEL TOLIMA",
#           "INSTITUTO DEPARTAMENTAL DE SALUD DE NORTE DE SANTANDER",
#           "SECRETARIA DE SALUD DEPARTAMENTAL DE BOLIVAR",
#           "GOBERNACION DEL CHOCO", "DIRECCION TERRITORIAL DE SALUD DE CALDAS"
mask3 = df_Giros_1['CÓDIGO MUNICIPIO'].isna()
 
PATRON_ENTIDAD_DPTO = (
    r'DEPARTAMENTO\s+DE(?:L)?'
    r'|GOBIERNO\s+DEPARTAMENTAL'
    r'|GOBERNACION\s+DE(?:L)?'
    r'|INSTITUTO\s+DEPARTAMENTAL'
    r'|SECRETARIA\s+DE\s+SALUD\s+DEPARTAMENTAL'
    r'|DIRECCION\s+TERRITORIAL\s+DE\s+SALUD'
)
es_entidad_dpto = df_Giros_1.loc[mask3, 'Nombre Razon Social'].str.contains(
    PATRON_ENTIDAD_DPTO, na=False
)
idx3 = df_Giros_1.loc[mask3][es_entidad_dpto].index
 
davi_dpto_uniq = (df_Davipola[['_dpto_norm', 'CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO']]
                  .drop_duplicates(subset='_dpto_norm'))
 
df_paso3 = (df_Giros_1.loc[idx3, ['_dpto_desc']]
            .merge(davi_dpto_uniq,
                   left_on='_dpto_desc',
                   right_on='_dpto_norm',
                   how='left'))
 
df_Giros_1.loc[idx3, 'CÓDIGO DEPARTAMENTO'] = df_paso3['CÓDIGO DEPARTAMENTO'].values
df_Giros_1.loc[idx3, 'NOMBRE DEPARTAMENTO'] = df_paso3['NOMBRE DEPARTAMENTO'].values
# CÓDIGO MUNICIPIO / NOMBRE MUNICIPIO quedan NaN → correcto, son entidades de nivel dpto
 
n3 = df_Giros_1.loc[idx3, 'CÓDIGO DEPARTAMENTO'].notna().sum()
print(f"Paso 3 — entidades departamentales: {n3:,} filas con dpto asignado")
 
 
# ── PASO 4: Distritos especiales ────────────────────────────────
# Cartagena, Barranquilla, Santa Marta, Bogotá, etc.
# No siguen el patrón "MUNICIPIO DE X" ni aparecen en Davipola con ese nombre.
mask4 = df_Giros_1['CÓDIGO MUNICIPIO'].isna()
 
DISTRITOS = {
    'CARTAGENA':    ('13', 'BOLIVAR',         '13001', 'CARTAGENA DE INDIAS'),
    'BARRANQUILLA': ('08', 'ATLANTICO',       '08001', 'BARRANQUILLA'),
    'SANTA MARTA':  ('47', 'MAGDALENA',       '47001', 'SANTA MARTA'),
    'BOGOTA':       ('11', 'BOGOTA D.C.',     '11001', 'BOGOTA D.C.'),
    'RIOHACHA':     ('44', 'LA GUAJIRA',      '44001', 'RIOHACHA'),
    'BUENAVENTURA': ('76', 'VALLE DEL CAUCA', '76109', 'BUENAVENTURA'),
    'SAN ANDRES':   ('88', 'SAN ANDRES',      '88001', 'SAN ANDRES'),
}
 
def resolver_distrito(rs):
    rs_norm = normalizar_robusto(str(rs))
    for clave, vals in DISTRITOS.items():
        if clave in rs_norm:
            return vals
    return (None, None, None, None)
 
if mask4.any():
    tmp = df_Giros_1.loc[mask4, 'Nombre Razon Social'].apply(resolver_distrito)
    df_Giros_1.loc[mask4, 'CÓDIGO DEPARTAMENTO'] = [v[0] for v in tmp]
    df_Giros_1.loc[mask4, 'NOMBRE DEPARTAMENTO'] = [v[1] for v in tmp]
    df_Giros_1.loc[mask4, 'CÓDIGO MUNICIPIO']    = [v[2] for v in tmp]
    df_Giros_1.loc[mask4, 'NOMBRE MUNICIPIO']    = [v[3] for v in tmp]
    n4 = df_Giros_1.loc[mask4, 'CÓDIGO MUNICIPIO'].notna().sum()
    print(f"Paso 4 — distritos especiales:      {n4:,} nuevas filas cruzadas")
 

Paso 1 — nombre único:               38,549 filas cruzadas
Paso 2 — nombre + dpto Descripcion: 6,464 nuevas filas cruzadas
Paso 3 — entidades departamentales: 2,367 filas con dpto asignado
Paso 4 — distritos especiales:      279 nuevas filas cruzadas


In [9]:
# PRE-PASO – Normalizar Davipola + recalcular Nombre_match
# ═════════════════════════════════════════════════════════════════
# Orden obligatorio:
#   1. Normalizar columnas de texto de Davipola a ASCII (elimina tildes,
#      evita duplicados ATLÁNTICO/ATLANTICO, BOLÍVAR/BOLIVAR, etc.)
#   2. Recalcular _dpto_norm (ahora sí sin tildes)
#   3. Recalcular Nombre_match en Giros con limpiar_nombre corregida
#   4. Reconstruir davi_uniq / davi_mun_dpto / davi_dpto_uniq
#   5. Limpiar resultados ya asignados en Pasos 1-4
 
# 1 — Normalizar columnas de resultado en Davipola
for _c in ['NOMBRE DEPARTAMENTO', 'NOMBRE MUNICIPIO']:
    df_Davipola[_c] = df_Davipola[_c].apply(normalizar_robusto)
 
# 2 — Recalcular columnas de llave (dependen de las columnas ya normalizadas)
df_Davipola['Nombre_match'] = df_Davipola['NOMBRE MUNICIPIO'].apply(normalizar_robusto)
df_Davipola['_dpto_norm']   = df_Davipola['NOMBRE DEPARTAMENTO'].apply(normalizar_robusto)
 
# 3 — Recalcular Nombre_match en Giros con limpiar_nombre corregida
#     (reemplaza lo calculado en Cell 7 del notebook)
df_Giros_inicial['Nombre_match'] = df_Giros_inicial['Nombre Razon Social'].apply(limpiar_nombre)
 
# 4 — Reconstruir conjuntos de merge
davi_uniq = (df_Davipola[COLS_DAVI]
             .drop_duplicates(subset='Nombre_match', keep=False))
davi_mun_dpto = (df_Davipola[COLS_DAVI]
                 .drop_duplicates(subset=['Nombre_match', '_dpto_norm']))
davi_dpto_uniq = (df_Davipola[['_dpto_norm', 'CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO']]
                  .drop_duplicates(subset='_dpto_norm'))
 
# 5 — Limpiar tildes en resultados ya asignados por los Pasos 1-4
for _c in ['NOMBRE DEPARTAMENTO', 'NOMBRE MUNICIPIO']:
    if _c in df_Giros_1.columns:
        df_Giros_1[_c] = df_Giros_1[_c].apply(
            lambda x: normalizar_robusto(x) if pd.notna(x) else x
        )
 
print("PRE — Davipola normalizada, Nombre_match recalculado, conjuntos de merge reconstruidos.")
 
# ─────────────────────────────────────────────────────────────────
# UTILIDADES DE APOYO
# ─────────────────────────────────────────────────────────────────
 
def _strip_prefijo(texto):
    """
    Elimina prefijos de entidad territorial incluyendo variantes con typo.
    Devuelve el nombre "limpio" normalizado.
    """
    PREFIJOS_EXTENDIDOS = [
        # Prefijos con typo
        r'MINICIPIO\s+DE(?:L)?\s+',
        r'MINICIPIO\s+',
        r'MUNICIPO\s+DE(?:L)?\s+',
        r'MUNICIPO\s+',
        r'ALCADIA\s+MUNICIPAL\s+DE(?:L)?\s+',
        r'ALCADIA\s+MUNICIPAL\s+',
        r'ALCADIA\s+DE(?:L)?\s+',
        r'ALCADIA\s+',
        # Prefijos estándar (ya cubiertos por limpiar_nombre, pero los repetimos
        # para usar esta función de forma autónoma)
        r'MUNICIPIO\s+DE(?:L)?\s+',
        r'MUNICIPIO\s+',
        r'DEPARTAMENTO\s+ARCHIPIELAGO\s+DE\s+',
        r'DEPARTAMENTO\s+DE(?:L)?\s+',
        r'GOBIERNO\s+DEPARTAMENTAL\s+DE(?:L)?\s+',
        r'GOBERNACION\s+DE(?:L)?\s+',
        r'GOBERNACION\s+',
        r'ALCALDIA\s+MUNICIPAL\s+DE(?:L)?\s+',
        r'ALCALDIA\s+MUNICIPIO\s+',
        r'ALCALDIA\s+MUNICIPAL\s+',
        r'ALCALDIA\s+DE(?:L)?\s+',
        r'ALCALDIA\s+',
    ]
    for p in PREFIJOS_EXTENDIDOS:
        texto = re.sub(p, '', texto).strip()
 
    SUFIJOS_LIMPIEZA = [
        r'\s+DPTO\.?\s*DE(?:L)?\s+\w+\.?$',
        r'\s+DEPARTAMENTO\s+DE(?:L)?\s+\w+\.?$',
        r'\s+ALCALDIA\s+MUNICIPAL\.?$',
        r'\s+ALCALDIA\.?$',
        r'\s+TESORERIA$',
        r'\s+EN\s+REESTRUCTURACION.*$',
        r'\.$',
        r'\s+-\s+\w+$',           # "NOMBRE - ANTIOQUIA"  →  "NOMBRE"
    ]
    for s in SUFIJOS_LIMPIEZA:
        texto = re.sub(s, '', texto).strip()
 
    return texto
 
 
DPTOS_SUFIJO = {
    'ANTIOQUIA', 'ATLANTICO', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA',
    'CASANARE', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAMARCA',
    'GUAINIA', 'GUAVIARE', 'HUILA', 'MAGDALENA', 'META', 'NARINO',
    'PUTUMAYO', 'QUINDIO', 'RISARALDA', 'AMAZONAS', 'SANTANDER',
    'SUCRE', 'TOLIMA', 'VAUPES', 'VICHADA', 'ARAUCA',
    'VALLE', 'NORTE DE SANTANDER',
}
 
 
def _quitar_sufijo_dpto(nombre):
    """
    Si la última(s) palabras del nombre coinciden con un departamento conocido,
    las elimina y las devuelve por separado.
    Retorna (nombre_limpio, dpto_detectado_o_None).
    """
    partes = nombre.split()
 
    # Probar sufijos de 3 palabras (ej. NORTE DE SANTANDER)
    if len(partes) >= 4:
        sufijo3 = ' '.join(partes[-3:])
        if sufijo3 in DPTOS_SUFIJO:
            return ' '.join(partes[:-3]).strip(), sufijo3
 
    # Sufijos de 2 palabras (ej. VALLE DEL CAUCA → 'VALLE DEL CAUCA' no está en set,
    # pero 'VALLE' sí; esto se resuelve con el caso 1 palabra)
    if len(partes) >= 3:
        sufijo2 = ' '.join(partes[-2:])
        if sufijo2 in DPTOS_SUFIJO:
            return ' '.join(partes[:-2]).strip(), sufijo2
 
    # Sufijos de 1 palabra
    if len(partes) >= 2 and partes[-1] in DPTOS_SUFIJO:
        return ' '.join(partes[:-1]).strip(), partes[-1]
 
    return nombre, None
 
 
# ─────────────────────────────────────────────────────────────────
# MAPA DE CORRECCIONES MANUALES
# Nombre limpio (normalizado) → nombre en Davipola (normalizado)
# ─────────────────────────────────────────────────────────────────
CORRECCIONES_MANUALES = {
    # ── Typos ortográficos ────────────────────────────────────────
    'SAN FRANSISCO':                    'SAN FRANCISCO',
    'VILLPINZON':                       'VILLAPINZON',
    'LACAPILLA':                        'LA CAPILLA',
    'MAGUIPAYAN':                       'MAGUIPAYANA',   # nombre en Davipola
 
    # ── Casos confirmados por usuario (Giros → Davipola) ─────────
    # 1.  MUNICIPIO DE MOMPOS → SANTA CRUZ DE MOMPOX
    'MOMPOS':                           'SANTA CRUZ DE MOMPOX',
    # 2.  ALCALDIA MUNICIPAL DE ARMERO GUAYABAL → ARMERO
    'ARMERO GUAYABAL':                  'ARMERO',
    # 3.  MUNICIPIO DE PACHO ALCALDIA MUNICIPAL → PACHO
    'PACHO':                            'PACHO',          # ya correcto; sufijo limpiado antes
    # 4.  MUNICIPIO DE PUERTO SANTANDER → PUERTO SANTANDER (Norte de Santander)
    #     (ambiguo: existe también en Vichada → se resuelve con dpto en el merge)
    'PUERTO SANTANDER':                 'PUERTO SANTANDER',
    # 5.  MUNICIPIO DEL PEÑON → EL PEÑON (Bolívar)
    'PENON':                            'EL PENON',
    'PEÑON':                            'EL PENON',
    # 6.  MUNICIPIO DE PUERTO NARIÑO → PUERTO NARINO
    'PUERTO NARINO':                    'PUERTO NARINO',
    # 7.  MUNICIPIO NUEVA CARAMANTA → CARAMANTA
    'NUEVA CARAMANTA':                  'CARAMANTA',
    # 8.  MUNICIPIO DE TOLUVIEJO → SAN JOSE DE TOLUVIEJO
    'TOLUVIEJO':                        'SAN JOSE DE TOLUVIEJO',
    # 9.  MUNICIPIO DE TOLU EN REESTRUCTURACION → SANTIAGO DE TOLU
    #     (el sufijo "EN REESTRUCTURACION" se elimina en _strip_prefijo)
    'TOLU':                             'SANTIAGO DE TOLU',
    # 10. MUNICIPIO EL PITAL → PITAL
    'EL PITAL':                         'PITAL',
    # 11. MUNICIPIO DE SANTA FE DE ANTIOQUIA → SANTA FE DE ANTIOQUIA
    'SANTA FE DE ANTIOQUIA':            'SANTA FE DE ANTIOQUIA',
    # 12. MUNICIPIO DE UBATE → VILLA DE SAN DIEGO DE UBATE
    'UBATE':                            'VILLA DE SAN DIEGO DE UBATE',
    # 13. MUNICIPIO DE MAGANGUE ALCALDIA MUNICIPAL → MAGANGUE
    #     (sufijo "ALCALDIA MUNICIPAL" limpiado en _strip_prefijo)
    'MAGANGUE':                         'MAGANGUE',
    # 14. MUNICIPIO DE PUERTO BOYACA → PUERTO BOYACA
    'PUERTO BOYACA':                    'PUERTO BOYACA',
    # 15. MUNICIPIO DE CIUDAD BOLIVAR → CIUDAD BOLIVAR
    #     (ambiguo Antioquia/Bolívar → dpto desambigua)
    'CIUDAD BOLIVAR':                   'CIUDAD BOLIVAR',
    # 16. MUNICIPIO DE SAN JUAN DEL CESAR → SAN JUAN DEL CESAR
    'SAN JUAN DEL CESAR':               'SAN JUAN DEL CESAR',
    # 17. MUNICIPIO DE SINCE → SAN LUIS DE SINCE
    'SINCE':                            'SAN LUIS DE SINCE',
    # 18. MUNICIPIO DE SAN JOSE DEL GUAVIARE → SAN JOSE DEL GUAVIARE
    'SAN JOSE DEL GUAVIARE':            'SAN JOSE DEL GUAVIARE',
    # 19 & 20. SAN JACINTO DEL CAUCA (con y sin ALCALDIA)
    'SAN JACINTO DEL CAUCA':            'SAN JACINTO DEL CAUCA',
 
    # ── Otras variantes conocidas ─────────────────────────────────
    'CALIMA EL DARIEN':                 'CALIMA',
    'COLON GENOVA':                     'COLON',           # Nariño; dpto desambigua vs Putumayo
    'SANTACRUZ GUACHAVES':              'SANTA CRUZ',
    'MANAURE BALCON DEL CESAR':         'MANAURE',         # único Manaure en Cesar
    'CANTON DE EL SAN PABLO':           'EL CANTON DEL SAN PABLO',
    'OLAYA DE HERRERA':                 'OLAYA HERRERA',
    'SAN JUAN BAUTISTA DE GUACARI':     'GUACARI',
    'CAROLINA DEL PRINCIPE':            'CAROLINA',
    'LA UNION PANAMERICANA':            'UNION PANAMERICANA',
    'SANTA ROSA DEL SUR DE BOLIVAR':    'SANTA ROSA DEL SUR',
    'SALAZAR DE LAS PALMAS':            'SALAZAR',
    'PAZ DE RIO':                       'PAZ DE RIO',      # Boyacá — nombre exacto en Davipola
    'RIOVIEJO':                         'RIO VIEJO',
    'VILLA ROSARIO':                    'VILLA DEL ROSARIO',
    'MARIA LABAJA':                     'MARIA LA BAJA',
    'PIENDAMO':                         'PIENDAMO TUNIA',
    'CUASPUD':                          'CUASPUD CARLOSAMA',
    'RICAURTE':                         'RICAURTE',        # ambiguo → dpto desambigua
    'OBANDO':                           'OBANDO',
    'CARMEN DE APICALA':                'CARMEN DE APICALA',
    'CARMEN DE DARIEN':                 'CARMEN DEL DARIEN',
    'BUGA':                             'GUADALAJARA DE BUGA',
    'GUICAN':                           'GUICAN DE LA SIERRA',
    'LITORAL DEL SAN JUAN':             'LITORAL DEL SAN JUAN',
    'MOLINO':                           'EL MOLINO',
    'SOTARA':                           'SOTARA PAISPAMBA',
    'TABLON DE GOMEZ':                  'EL TABLON DE GOMEZ',
    'TABLON':                           'EL TABLON DE GOMEZ',
    'MARIQUITA':                        'SAN SEBASTIAN DE MARIQUITA',
    'PURISIMA':                         'PURISIMA DE LA CONCEPCION',
    'TUMACO':                           'SAN ANDRES DE TUMACO',
    'VILLA DE LEIVA':                   'VILLA DE LEYVA',
    'BETULIA':                          'BETULIA',         # ambiguo → dpto desambigua
    'NOROSI':                           'NOROSI',
    'HATILLO DE LOBA':                  'HATILLO DE LOBA',
 
    # ── Entidades sin municipio (nivel departamental) ─────────────
    'FONDO FINANCIERO DISTRITAL DE SALUD':              '__BOGOTA_DC__',
    'UNIDAD ADMINISTRATIVA ESPECIAL DE SALUD DE ARAUCA':'__DPTO_ARAUCA__',
    'UNIDADADMINISTRATIVAESPECIALDESALUDDEARAUCA':       '__DPTO_ARAUCA__',
}
 
# Entidades especiales nivel-dpto que el Paso 3 no captura
ENTIDADES_DPTO_EXTRA = {
    # Patrón en nombre normalizado → (cod_dpto, nom_dpto)
    'FONDO FINANCIERO DISTRITAL DE SALUD': ('11', 'BOGOTA D.C.'),
    'UNIDAD ADMINISTRATIVA ESPECIAL DE SALUD DE ARAUCA': ('81', 'ARAUCA'),
}
 
# ─────────────────────────────────────────────────────────────────
# PASO 5 – Typos en prefijo (MINICIPIO, MUNICIPO, ALCADIA)
# ─────────────────────────────────────────────────────────────────
mask5 = df_Giros_1['CÓDIGO MUNICIPIO'].isna() & df_Giros_1['CÓDIGO DEPARTAMENTO'].isna()
 
PATRON_TYPO = r'^(?:MINICIPIO|MUNICIPO|ALCADIA)\b'
idx5 = df_Giros_1.loc[mask5].index[
    df_Giros_1.loc[mask5, 'Nombre Razon Social']
    .str.upper().str.contains(PATRON_TYPO, na=False)
]
 
# Recalcular Nombre_match con prefijos con typo
df_Giros_1['_nm2'] = df_Giros_1['Nombre Razon Social'].apply(
    lambda x: normalizar_robusto(_strip_prefijo(normalizar_robusto(str(x))))
)
 
df_paso5 = (df_Giros_1.loc[idx5, ['_nm2', '_dpto_desc']]
            .merge(davi_mun_dpto,
                   left_on=['_nm2', '_dpto_desc'],
                   right_on=['Nombre_match', '_dpto_norm'],
                   how='left'))
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[idx5, col] = df_paso5[col].values
 
# Intento sin dpto para los que quedaron sin cruzar
still5 = df_Giros_1.loc[idx5][df_Giros_1.loc[idx5, 'CÓDIGO MUNICIPIO'].isna()].index
df_paso5b = (df_Giros_1.loc[still5, ['_nm2']]
             .merge(davi_uniq, left_on='_nm2', right_on='Nombre_match', how='left'))
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[still5, col] = df_paso5b[col].values
 
n5 = df_Giros_1.loc[idx5, 'CÓDIGO MUNICIPIO'].notna().sum()
print(f"Paso 5 — typos prefijo:              {n5:,} nuevas filas cruzadas")
 
 
# ─────────────────────────────────────────────────────────────────
# PASO 6 – "MUNICIPIO <NOMBRE>" sin "DE" + sufijo de departamento
# ─────────────────────────────────────────────────────────────────
mask6 = df_Giros_1['CÓDIGO MUNICIPIO'].isna() & df_Giros_1['CÓDIGO DEPARTAMENTO'].isna()
 
# Construir nombre_match limpio con _nm2 (ya calculado arriba)
# _nm2 ya tiene prefijo quitado; ahora quitar sufijo de dpto
 
def limpiar_con_sufijo_dpto(row):
    nombre = row['_nm2']
    nombre_sin_dpto, dpto_detectado = _quitar_sufijo_dpto(nombre)
    return pd.Series({'_nm3': nombre_sin_dpto,
                      '_dpto_sufijo': normalizar_robusto(dpto_detectado) if dpto_detectado else ''})
 
tmp6 = df_Giros_1.loc[mask6].apply(limpiar_con_sufijo_dpto, axis=1)
df_Giros_1.loc[mask6, '_nm3'] = tmp6['_nm3']
df_Giros_1.loc[mask6, '_dpto_sufijo'] = tmp6['_dpto_sufijo']
 
# Resolver dpto: priorizar sufijo del nombre, luego _dpto_desc
df_Giros_1.loc[mask6, '_dpto_para_match'] = df_Giros_1.loc[mask6, '_dpto_sufijo'].where(
    df_Giros_1.loc[mask6, '_dpto_sufijo'] != '', df_Giros_1.loc[mask6, '_dpto_desc']
)
 
idx6 = df_Giros_1.loc[mask6].index
 
# Intento 1: con dpto
df_paso6a = (df_Giros_1.loc[idx6, ['_nm3', '_dpto_para_match']]
             .merge(davi_mun_dpto,
                    left_on=['_nm3', '_dpto_para_match'],
                    right_on=['Nombre_match', '_dpto_norm'],
                    how='left'))
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[idx6, col] = df_paso6a[col].values
 
# Intento 2: sin dpto (nombre único)
still6 = df_Giros_1.loc[idx6][df_Giros_1.loc[idx6, 'CÓDIGO MUNICIPIO'].isna()].index
df_paso6b = (df_Giros_1.loc[still6, ['_nm3']]
             .merge(davi_uniq, left_on='_nm3', right_on='Nombre_match', how='left'))
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[still6, col] = df_paso6b[col].values
 
n6 = df_Giros_1.loc[idx6, 'CÓDIGO MUNICIPIO'].notna().sum()
print(f"Paso 6 — sin 'DE' + sufijo dpto:     {n6:,} nuevas filas cruzadas")
 
 
# ─────────────────────────────────────────────────────────────────
# PASO 7 – Correcciones manuales de nombres con variantes conocidas
# ─────────────────────────────────────────────────────────────────
mask7 = df_Giros_1['CÓDIGO MUNICIPIO'].isna() & df_Giros_1['CÓDIGO DEPARTAMENTO'].isna()
idx7  = df_Giros_1.loc[mask7].index
 
# Construir llave corregida
def aplicar_correccion(nm):
    return CORRECCIONES_MANUALES.get(nm, nm)
 
df_Giros_1.loc[idx7, '_nm_corr'] = df_Giros_1.loc[idx7, '_nm3'].apply(aplicar_correccion)
 
# Excluir marcadores especiales (__BOGOTA_DC__, __DPTO_ARAUCA__)
mask_especial = df_Giros_1.loc[idx7, '_nm_corr'].str.startswith('__')
idx7_norm   = idx7[~mask_especial]
idx7_esp    = idx7[mask_especial]
 
# Match con dpto
df_paso7a = (df_Giros_1.loc[idx7_norm, ['_nm_corr', '_dpto_para_match']]
             .merge(davi_mun_dpto,
                    left_on=['_nm_corr', '_dpto_para_match'],
                    right_on=['Nombre_match', '_dpto_norm'],
                    how='left'))
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[idx7_norm, col] = df_paso7a[col].values
 
# Match sin dpto (nombre único en Colombia)
still7 = df_Giros_1.loc[idx7_norm][df_Giros_1.loc[idx7_norm, 'CÓDIGO MUNICIPIO'].isna()].index
df_paso7b = (df_Giros_1.loc[still7, ['_nm_corr']]
             .merge(davi_uniq, left_on='_nm_corr', right_on='Nombre_match', how='left'))
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[still7, col] = df_paso7b[col].values
 
# Entidades especiales (Bogotá D.C., Arauca)
for idx_e in idx7_esp:
    marcador = df_Giros_1.loc[idx_e, '_nm_corr']
    if marcador == '__BOGOTA_DC__':
        df_Giros_1.loc[idx_e, 'CÓDIGO DEPARTAMENTO'] = '11'
        df_Giros_1.loc[idx_e, 'NOMBRE DEPARTAMENTO'] = 'BOGOTA D.C.'
    elif marcador == '__DPTO_ARAUCA__':
        df_Giros_1.loc[idx_e, 'CÓDIGO DEPARTAMENTO'] = '81'
        df_Giros_1.loc[idx_e, 'NOMBRE DEPARTAMENTO'] = 'ARAUCA'
 
n7 = df_Giros_1.loc[idx7, 'CÓDIGO MUNICIPIO'].notna().sum()
n7_dpto = (df_Giros_1.loc[idx7_esp, 'CÓDIGO DEPARTAMENTO'].notna().sum())
print(f"Paso 7 — correcciones manuales:      {n7:,} municipios  |  {n7_dpto:,} solo-dpto")
 
 
# ─────────────────────────────────────────────────────────────────
# PASO 8 – Entidades departamentales ampliadas
#   Captura GOBERNACION, GOBIERNO DEPARTAMENTAL, UNIDAD ADTVA.,
#   FONDO, INSTITUTO, SECRETARIA que no cayeron en Paso 3
# ─────────────────────────────────────────────────────────────────
mask8 = df_Giros_1['CÓDIGO DEPARTAMENTO'].isna()
 
PATRON_DPTO_AMP = (
    r'DEPARTAMENTO\s+DE(?:L)?'
    r'|GOBIERNO\s+DEPARTAMENTAL'
    r'|GOBERNACION\s+DE(?:L)?'
    r'|GOBERNACION\b'
    r'|INSTITUTO\s+DEPARTAMENTAL'
    r'|SECRETARIA\s+DE\s+SALUD\s+DEPARTAMENTAL'
    r'|DIRECCION\s+TERRITORIAL\s+DE\s+SALUD'
    r'|UNIDAD\s+ADMINISTRATIVA\s+ESPECIAL\s+DE\s+SALUD'
)
es_dpto8 = df_Giros_1.loc[mask8, 'Nombre Razon Social'].apply(
    lambda x: bool(re.search(PATRON_DPTO_AMP, normalizar_robusto(str(x))))
)
idx8 = df_Giros_1.loc[mask8][es_dpto8].index
 
# Extraer dpto del nombre normalizado
def extraer_dpto_de_nombre(rs):
    s = normalizar_robusto(str(rs))
    # "GOBERNACION DEL META" → "META", "GOBIERNO DEPARTAMENTAL DEL TOLIMA" → "TOLIMA"
    m = re.search(r'(?:DE(?:L)?\s+)([A-Z][A-Z\s]+)$', s)
    if m:
        cand = m.group(1).strip()
        # Correcciones
        correc = {
            'NORTE DE SANTANDER': 'NORTE DE SANTANDER',
            'CHOCO': 'CHOCO',
            'BOLIVAR': 'BOLIVAR',
            'NARINO': 'NARINO',
            'LA GUAJIRA': 'LA GUAJIRA',
            'VALLE DEL CAUCA': 'VALLE DEL CAUCA',
            'CALDAS': 'CALDAS',
        }
        return correc.get(cand, cand)
    return ''
 
df_Giros_1.loc[idx8, '_dpto_rs'] = df_Giros_1.loc[idx8, 'Nombre Razon Social'].apply(
    extraer_dpto_de_nombre
)
# Preferir _dpto_desc si ya existe
df_Giros_1.loc[idx8, '_dpto_final8'] = df_Giros_1.loc[idx8, '_dpto_desc'].where(
    df_Giros_1.loc[idx8, '_dpto_desc'] != '', df_Giros_1.loc[idx8, '_dpto_rs']
)
 
df_paso8 = (df_Giros_1.loc[idx8, ['_dpto_final8']]
            .merge(davi_dpto_uniq,
                   left_on='_dpto_final8',
                   right_on='_dpto_norm',
                   how='left'))
df_Giros_1.loc[idx8, 'CÓDIGO DEPARTAMENTO'] = df_paso8['CÓDIGO DEPARTAMENTO'].values
df_Giros_1.loc[idx8, 'NOMBRE DEPARTAMENTO'] = df_paso8['NOMBRE DEPARTAMENTO'].values
 
n8 = df_Giros_1.loc[idx8, 'CÓDIGO DEPARTAMENTO'].notna().sum()
print(f"Paso 8 — entidades dpto ampliadas:   {n8:,} filas con dpto asignado")
 
 
# ─────────────────────────────────────────────────────────────────
# PASO 9 – Match difuso (último recurso)
# Solo para filas sin municipio Y sin departamento
# ─────────────────────────────────────────────────────────────────
mask9 = df_Giros_1['CÓDIGO MUNICIPIO'].isna() & df_Giros_1['CÓDIGO DEPARTAMENTO'].isna()
idx9  = df_Giros_1.loc[mask9].index
 
davi_nombres = df_Davipola['Nombre_match'].tolist()
 
def match_difuso(nm, dpto_hint=''):
    """
    Busca el municipio más parecido en Davipola.
    Si hay dpto_hint, filtra primero por ese departamento.
    Umbral: 80 % de similitud (0.8).
    """
    if not nm:
        return None, None, None, None
 
    candidatos = df_Davipola
    if dpto_hint:
        sub = df_Davipola[df_Davipola['_dpto_norm'] == dpto_hint]
        if not sub.empty:
            candidatos = sub
 
    matches = get_close_matches(nm, candidatos['Nombre_match'].tolist(), n=1, cutoff=0.80)
    if not matches:
        return None, None, None, None
 
    fila = candidatos[candidatos['Nombre_match'] == matches[0]].iloc[0]
    return (fila['CÓDIGO DEPARTAMENTO'], fila['NOMBRE DEPARTAMENTO'],
            fila['CÓDIGO MUNICIPIO'],    fila['NOMBRE MUNICIPIO'])
 
resultados9 = df_Giros_1.loc[idx9].apply(
    lambda r: pd.Series(
        match_difuso(r.get('_nm_corr') or r.get('_nm3') or r.get('_nm2') or r['Nombre_match'],
                     r.get('_dpto_para_match', '') or r.get('_dpto_desc', '')),
        index=['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
               'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']
    ), axis=1
)
for col in ['CÓDIGO DEPARTAMENTO', 'NOMBRE DEPARTAMENTO',
            'CÓDIGO MUNICIPIO', 'NOMBRE MUNICIPIO']:
    df_Giros_1.loc[idx9, col] = resultados9[col].values
 
n9 = df_Giros_1.loc[idx9, 'CÓDIGO MUNICIPIO'].notna().sum()
print(f"Paso 9 — match difuso:               {n9:,} nuevas filas cruzadas")
 
 
# ─────────────────────────────────────────────────────────────────
# LIMPIEZA DE COLUMNAS AUXILIARES
# ─────────────────────────────────────────────────────────────────
cols_aux = ['_dpto_desc', 'Nombre_match', '_nm2', '_nm3', '_nm_corr',
            '_dpto_sufijo', '_dpto_para_match', '_dpto_rs', '_dpto_final8']
df_Giros_1.drop(columns=[c for c in cols_aux if c in df_Giros_1.columns],
                inplace=True, errors='ignore')
 
 
# ─────────────────────────────────────────────────────────────────
# RESUMEN FINAL
# ─────────────────────────────────────────────────────────────────
total_ok   = df_Giros_1['CÓDIGO MUNICIPIO'].notna().sum()
total_dpto = (df_Giros_1['CÓDIGO MUNICIPIO'].isna() &
              df_Giros_1['CÓDIGO DEPARTAMENTO'].notna()).sum()
total_nok  = (df_Giros_1['CÓDIGO MUNICIPIO'].isna() &
              df_Giros_1['CÓDIGO DEPARTAMENTO'].isna()).sum()
 
print(f"\n{'─'*60}")
print(f"Total filas:                  {len(df_Giros_1):,}")
print(f"Con municipio:                {total_ok:,}")
print(f"Solo departamento (correcto): {total_dpto:,}")
print(f"Sin municipio ni dpto:        {total_nok:,}  ← revisar manualmente")
 
if total_nok > 0:
    print("\nTop sin resolver:")
    print(df_Giros_1[
        df_Giros_1['CÓDIGO MUNICIPIO'].isna() &
        df_Giros_1['CÓDIGO DEPARTAMENTO'].isna()
    ]['Nombre Razon Social'].value_counts().head(30).to_string())

PRE — Davipola normalizada, Nombre_match recalculado, conjuntos de merge reconstruidos.
Paso 5 — typos prefijo:              87 nuevas filas cruzadas
Paso 6 — sin 'DE' + sufijo dpto:     57 nuevas filas cruzadas
Paso 7 — correcciones manuales:      1,407 municipios  |  114 solo-dpto
Paso 8 — entidades dpto ampliadas:   2,398 filas con dpto asignado
Paso 9 — match difuso:               678 nuevas filas cruzadas

────────────────────────────────────────────────────────────
Total filas:                  50,065
Con municipio:                47,521
Solo departamento (correcto): 2,512
Sin municipio ni dpto:        32  ← revisar manualmente

Top sin resolver:
Nombre Razon Social
MUNICIPIO MAGUIPAYAN    32


In [10]:
#  RESULTADO FINAL
df_Giros_1 = df_Giros_1[columnas_deseadas]
 
total_ok  = df_Giros_1['CÓDIGO MUNICIPIO'].notna().sum()
total_nok = df_Giros_1['CÓDIGO MUNICIPIO'].isna().sum()
 
print(f"\n{'─'*55}")
print(f"Total filas:       {len(df_Giros_1):,}")
print(f"Con municipio:     {total_ok:,}")
print(f"Sin municipio:     {total_nok:,}  ← solo entidades nivel departamental")
 
if total_nok > 0:
    print("\nTop sin municipio (esperado: solo entidades departamentales):")
    print(df_Giros_1[df_Giros_1['CÓDIGO MUNICIPIO'].isna()]['Nombre Razon Social']
          .value_counts().head(20).to_string())


───────────────────────────────────────────────────────
Total filas:       50,065
Con municipio:     47,521
Sin municipio:     2,544  ← solo entidades nivel departamental

Top sin municipio (esperado: solo entidades departamentales):
Nombre Razon Social
GOBIERNO DEPARTAMENTAL DEL TOLIMA                         89
SECRETARIA DE SALUD DEPARTAMENTAL DE BOLIVAR              89
DEPARTAMENTO DE LA GUAJIRA                                89
DEPARTAMENTO DEL META                                     89
INSTITUTO DEPARTAMENTAL DE SALUD DE NARINO                89
DEPARTAMENTO DEL CESAR                                    89
DEPARTAMENTO DEL HUILA                                    89
DEPARTAMENTO DEL CAQUETA                                  89
DEPARTAMENTO DEL QUINDIO                                  89
GOBERNACION DEL CHOCO                                     89
DEPARTAMENTO DEL VAUPES                                   89
DIRECCION TERRITORIAL DE SALUD DE CALDAS                  89
DEPARTAMENTO 

In [11]:
# Descargar Datos df_Giros_1
df_Giros_1.head(10)

,Fecha de pago,CÓDIGO DEPARTAMENTO,NOMBRE DEPARTAMENTO,CÓDIGO MUNICIPIO,NOMBRE MUNICIPIO,Identificacion- NIT,Nombre Razon Social,Descripcion,Fuente,Valor Girado,Concepto Pago,Ordenes de Pago
0,2015-01-29,81,ARAUCA,81300,FORTUL,800136069,MUNICIPIO DE FORTUL,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,27.047.767,GIRO SGP SALUD PUBLICA VIG 2014,9207515
1,2015-01-29,63,QUINDIO,63001,ARMENIA,890000464,MUNICIPIO DE ARMENIA,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,130.243.583,GIRO SGP SALUD PUBLICA VIG 2014,9207615
2,2015-01-29,25,CUNDINAMARCA,25740,SIBATE,899999372,MUNICIPIO DE SIBATE,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,12.600.367,GIRO SGP SALUD PUBLICA VIG 2014,9207715
3,2015-01-29,05,ANTIOQUIA,05088,BELLO,890980112,MUNICIPIO DE BELLO,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,214.941.350,GIRO SGP SALUD PUBLICA VIG 2014,9207815
4,2015-01-29,05,ANTIOQUIA,05858,VEGACHI,890985285,MUNICIPIO DE VEGACHI,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,9.040.907,GIRO SGP SALUD PUBLICA VIG 2014,9207915
5,2015-01-29,05,ANTIOQUIA,05890,YOLOMBO,890984030,MUNICIPIO DE YOLOMBO ANTIOQUIA,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,16.432.107,GIRO SGP SALUD PUBLICA VIG 2014,9208015
6,2015-01-29,25,CUNDINAMARCA,25001,AGUA DE DIOS,890680149,MUNICIPIO DE AGUA DE DIOS,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,5.366.788,GIRO SGP SALUD PUBLICA VIG 2014,9208115
7,2015-01-29,25,CUNDINAMARCA,25200,COGUA,899999466,MUNICIPIO DE COGUA,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,6.625.193,GIRO SGP SALUD PUBLICA VIG 2014,9208215
8,2015-01-29,73,TOLIMA,NaN,NaN,800113672,GOBIERNO DEPARTAMENTAL DEL TOLIMA,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,660.202.336,GIRO SGP SALUD PUBLICA VIG 2014,9208315
9,2015-01-29,13,BOLIVAR,NaN,NaN,890480126,SECRETARIA DE SALUD DEPARTAMENTAL DE BOLIVAR,SISTEMA GENERAL DE PARTICIPACIONES SALUD PUBLI...,Nación,1.155.163.350,GIRO SGP SALUD PUBLICA VIG 2014,9208415


In [12]:
os.makedirs('/kaggle/working/Giros_Procesados', exist_ok=True)

df_Giros_1.to_parquet('/kaggle/working/Giros_Procesados/Giros_final.parquet', index=False)
df_Giros_1.to_csv('/kaggle/working/Giros_Procesados/Giros_final.csv', index=False)
df_Giros_1.to_excel('/kaggle/working/Giros_Procesados/Giros_final.xlsx', index=False)

print("Listo")

Listo
